# `C09` — Price-incremented icebergs

Scenario **C09** of the Track A certification ladder (card `OMIE-CERT-C09-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C9). Oracle kind: **exact-parametric**. **High-risk scenario** (independence rule 4: the sign rule). **Run r2** (`OMIE-CERT-RERUN-1`, K7-U verdict REPEAT at K2, log entry 20): the expectation below is re-authored from the specification only.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. Every pass/fail assertion here is an **exact integer equality** (or membership in an explicitly listed set) on order ids, prices in ticks (0.01 EUR/MWh), quantities in ticks (0.1 MW) and submission indices. Euro figures are shown, never asserted.

## 1. Purpose

**Purpose.** When an iceberg carries a price increment, each regenerated slice rests at a price offset
from the last, so one resting order walks the price ladder as it is consumed — and the iceberg, not
the aggressor, sets every one of those prices. The scenario earns its own place because of a specific
nasty failure: a **sign error on the increment**. Flip the sign and the slices drift the wrong way,
the engine still trades, volume still conserves, cash still balances, and every invariant passes.
Only an assertion on the exact price *sequence* catches it. It also pins where matching stops — at
the first slice whose drifted price no longer crosses — leaving the iceberg resting with hidden
quantity, which an agent would otherwise misread as exhausted liquidity.

One constraint that is easy to get backwards. §13.6 makes the **sign** rule hard (buy increments
negative, sell positive) and explicitly establishes that there is **no rule-level magnitude bound**,
citing a real `PPD = -3.00` counter-example (§13.6 writes it with an ASCII hyphen; the notebook byte-copies it). So draws must respect the sign rule, and the draw
*range* must be justified as an empirical test envelope — never cited as though it were a bound.

**A known finding, recorded in advance.** The engine does **not** enforce the sign rule: the validator
checks only `0 < peak ≤ quantity` (`order_validator.py`), and a wrong-sign increment is accepted
(`open-questions.md`, OQ-23, CORRECTION + CLOSURE NOTE (a)). So the scenario's main draws are
sign-valid by construction, and one extra, clearly labelled draw submits a wrong-sign increment as an
**observation**: its accept/reject outcome is logged, never asserted, because a locked contract must not
pin behaviour on a domain-forbidden input (that same note). It is the evidence for a named
**conformance gap** in the certificate, and it is excluded from the pass criterion. Fixing it is outside this ladder.

*(Verbatim from `scenario-ladder.md` §C9.)*

**Correction for this run (r2; `OMIE-CERT-RERUN-1`, applied exactly as filed).** *"C09 — the wrong-sign draw becomes an asserted rejection (after S1); restate the scope (one iceberg, no competing level)."* The third paragraph of the purpose above is **superseded** for this run. `market-spec.md` v2.14 §5.4 now lists the ICEwPI price-increment sign as an **order rejection condition** (a BUY with PPD > 0, or a SELL with PPD < 0, is rejected; OMIE Rules Annex §4.5), and `OMIE-ICEWPI-SIGN-REJECT-1` (S1) is the engine change that makes the validator apply it. So the wrong-sign draws are no longer an observation of a conformance gap: each is an **asserted rejection** — the iceberg is **REJECTED**, **no trade** is written, and the **book is unchanged** by its submission. Pinning that outcome pins a market rule (§5.4), not behaviour on an input whose outcome the rules leave open (finding F8). The canonical §C9 text still carries the superseded paragraph (finding F13).

**Scope (restated).** Every draw is **one contract, one iceberg and one aggressor, and nothing else**: there is **no competing price level** and no second resting order on either side. So nothing can interleave with the iceberg's ladder, and the order in which the aggressor meets the slices is the ladder itself. This is deliberate: the scenario isolates the **drift** (price sequence and stop index) and the **sign rule**. What a competing level does to that order (§4.2.1 Rule 2 *Re-ranking*, v2.14) is outside C09 (finding F11).

**Certifies (r2):** ICEwPI slice pricing on sign-valid draws, with one iceberg and no competing level (§4.2); §4.2.1 Rule 2 (stop at the first non-crossing drifted slice); §6.2 item 7 (the iceberg is always the price-setter); the ICEwPI sign rule as an order **rejection** condition, on both sides (§5.4 v2.14; §13.6 sign rule, item 1; §5.1 REJECTED; §5.2: a rejected order never enters the book, so it takes no timestamp); §13.6's **magnitude** statement only (no rule-level bound) as the ground for the draw envelope, never cited as a bound. **Pass:** exact integer-tick equality on the full price sequence and the stop index of every valid main draw; every valid wrong-sign draw rejected with zero trades and an unchanged book, exactly as section 5 states; the eager and jitted runs agree; the full battery runs (on both constructions); I8 capacity headroom; I9 non-vacuity; every quote matches its source byte for byte. **Does not certify:** the re-ranking of a regenerated slice against a competing order (scope, F11); an **incoming** ICEwPI aggressor (§4.2.1 Rule 2 *Incoming iceberg* (6), OQ-34); the §5.4 *peak larger than total* condition; the rounding of a sub-tick PPD; the PPD of non-ICE orders; the validator's own reason code (unobserved; finding F15).

**How the claim is pinned.** Two constructions of the generator, each on a fresh book of one contract:

- **`C09.icewpi_drift`** (the main draw, `DRAWS` valid draws). Submission 0: a resting ICEwPI iceberg, side $\sigma$, base price $\mathit{base}$, sign-valid increment $\mathit{ppd} \ne 0$ ($\mathit{ppd} > 0$ for SELL, $< 0$ for BUY), peak $\mathit{peak}$, total $\mathit{total}$, $n = \lceil \mathit{total}/\mathit{peak} \rceil \ge 3$ slices. Submission 1: one NON aggressor on the other side whose limit $L$ crosses exactly the first $m \ge 1$ drifted slices, and whose quantity exceeds those $m$ slices, so matching stops **on price** at slice $m$ and the aggressor's residual rests.
- **`C09.icewpi_wrong_sign`** (`WRONG_DRAWS` valid draws, both sides). The same shape with the increment's sign flipped: a BUY with $\mathit{ppd} > 0$ or a SELL with $\mathit{ppd} < 0$. By §5.4 the iceberg is **rejected**; the aggressor, now facing an empty book, rests whole at $L$.

Per valid main draw, in one combined assertion whose expected and observed values are recorded **in full** (section 7): **(A)** acceptance; **(P)** the exact price sequence $\mathit{base}, \mathit{base}+\mathit{ppd}, \ldots, \mathit{base}+(m-1)\,\mathit{ppd}$ and the **stop index** $m$; **(Q)** the exact trade rows, `[buyer, seller, price, qty, submission]` (D3-08: the counterparty sequence is pinned); **(F)** filled and remaining quantity per order, no eviction or cancellation; **(H)** the iceberg left with positive hidden quantity; **(B)** the final book, one of two listed books (finding F3).

Per valid wrong-sign draw, in one combined assertion, recorded in full: **(R)** the iceberg rejected at submission 0 (a validator rejection), the aggressor accepted; **(Z)** zero trades; **(U)** the book unchanged by submission 0 — both sides, the trade log, the iceberg registry and the timestamp counter identical before and after it; **(F′)** filled 0 for both, remaining 0 for the iceberg (no hidden quantity) and $q_{agg}$ for the aggressor, nothing evicted or cancelled; **(B′)** the final book: the aggressor alone, $[[L,\ q_{agg},\ o_{agg}]]$ on its side, nothing on the iceberg's side.

Then the battery (RB, I1–I9) on every valid run of both constructions, with explicit assertions that I3 and I5 hold on every valid draw (a `not_applicable` status is vacuously satisfied, never a failure); the capacity checks (hop bound, largest slice count); the sign-discrimination check (finding F5); coverage of the four main classes and of both wrong-sign sides; the eager comparisons.

**K2 findings** (recorded here; none blocks the expectation):

- **F1 — spec version and the draw envelope.** The card names `market-spec.md` v2.12; the file is now **v2.14** (v2.13 touched §4.2.1 Rule 3 and §6.2 item 7, OQ-31; v2.14 is additive only — §5.4's two new rejection conditions, new §4.2.1 bullets, no byte of a quoted passage changed). Every quote is checked against the current file at run time. The draw envelope is the generator's: $|\mathit{ppd}| \in [1, 50]$ ticks (0.01–0.50 EUR/MWh), base 30–90 EUR/MWh, peak 1.0–8.0 MW, $n \in [3, 19]$. It is an **empirical test envelope** tied to the §13.6 sample (`orders_20250601.1`, $|\mathit{PPD}|$ up to about 1.55 EUR/MWh), **never a bound**: §13.6 and §5.4 say there is none, and it does not reach the 2026-01-12 `PPD = -3.00` counter-example. The envelope actually drawn is reported in section 4.
- **F2 — the sign rule: a precondition of the main draws, a claim of the wrong-sign draws.** Every main draw is checked from its tape to be sign-valid, and every wrong-sign draw to be sign-**invalid** with $\mathit{ppd} \ne 0$ (section 4). The drift is certified **given** a sign-valid increment (review N-1); the rejection is certified on the sign-invalid ones.
- **F3 — slice $m$ may be visible or not yet shown.** When slice $m-1$ is used up, §4.2 and Rule 2 activate slice $m$ at its drifted price $\mathit{base}+m\,\mathit{ppd}$, where it does not cross $L$ and rests. The §4.2.1 *Scope clarification* and OQ-20's ruling also permit the state in which a fully consumed peak's next slice has not yet been activated ("a permitted between-aggressor state"). The final book is therefore asserted to be **one of** two listed books: slice $m$ visible at **exactly** $\mathit{base}+m\,\mathit{ppd}$ with $\min(\mathit{peak}, \mathit{total}-m\,\mathit{peak})$, or no visible row for the iceberg; in both the aggressor's residual rests at $L$. In both, the iceberg's remaining quantity (visible + hidden) is exact, $\mathit{total} - m\,\mathit{peak}$, and its hidden quantity is positive. Which member occurred is reported per draw and counted. (The purpose's "misread as exhausted liquidity" is the hazard of the second member: the count is the evidence.) See F12 for v2.14's activation-timing note.
- **F4 — the price-setter claim discriminates on $k < m-1$ only.** By construction $L = \mathit{base}+(m-1)\,\mathit{ppd}$, so the last traded slice is priced exactly at $L$ and a price at $L$ cannot tell maker from aggressor there. On draws with $m = 1$ the price sequence is $[\mathit{base}] = [L]$: the price-setter is not discriminated, and a sign error is caught by the stop index alone (F5). The classes with $m \ge 2$ (where slices $0..m-2$ print strictly through $L$) are required by the coverage check.
- **F5 — a drift-sign error is caught on every main draw (non-vacuity of the oracle).** For each valid main draw section 5 also walks the **counterfactual** in which the engine applies the ladder with the increment's sign flipped (no validation: the hazard is a drift error, not a validator error) and asserts that its trade list differs from the expectation. It always does: the flipped slice 1, at $\mathit{base}-\mathit{ppd}$, lies on the far side of $\mathit{base}$ from the aggressor and crosses $L$, and the aggressor has more than $m\,\mathit{peak}$ to give, so a sign-flipped engine trades at least $m+1$ rows.
- **F6 — what "stop index" observes.** On this geometry each slice $k < m$ is full size and trades with the single aggressor in exactly one row, so the stop index is observed as the **number of trade rows** (all written in submission 1); together with the exact rows (Q) and the residual at $L$ (B), it says matching stopped at slice $m$, on price.
- **F7 — passive/aggressor labels are not asserted.** As in C08 (its F3): which leg the engine labels passive inside a multi-slice match is OQ-31's open engine-facing item (b); the rules fix the price-setter (the iceberg), not the label. Trades are pinned by buyer and seller order ids, price, quantity and submission; the number of hop-shaped rows (passive = the submitting order) is reported, not asserted. No Rule-3 hop is expected: slice $m$ does not cross the residual at $L$ (they are $|\mathit{ppd}| \ge 1$ tick apart), and I3 asserts the book is uncrossed.
- **F8 — the wrong-sign draw is an asserted rejection (r2 correction).** In r1 it was an observation, because OQ-23's CORRECTION + CLOSURE NOTE (a) forbids a locked contract from pinning behaviour on a domain-forbidden input, and the rules then named no outcome for it. v2.14 §5.4 now names the outcome: the order **is rejected**. Pinning that pins the rule, so the note's concern does not apply (the quote is kept for the record). With the iceberg rejected, the aggressor meets an empty book and rests whole; there is no hop, so review D3-04's "I5 `unruled_hop` expected" note no longer applies, and the wrong-sign battery is **folded** into the battery table like any other run. `WRONG_DRAWS` draws (not one) are asserted, so both sides are covered (coverage check).
- **F9 — a capacity flag inside the margin is a failure, not only a discard** (as C08 F7). A draw of either construction that passed the pre-run margin check (hop bound ≤ 160, ≤ 80 trades, ≤ 80 resting per side, ≤ 800 icebergs, every $\lceil \mathit{qty}/\mathit{peak} \rceil \le 64$) and still raised a capacity flag is discarded **and** recorded as a failed assertion.
- **F10 — what the engine reports is not asserted here.** The spec defines no per-submission reported price for a multi-price submission. Report-equals-tape is the battery's I6; it runs on every draw. To keep the results JSON compact, per-draw records carry the asserted values only.
- **F11 — scope: one iceberg, no competing level.** v2.14 adds §4.2.1 Rule 2 *Re-ranking*: each regenerated slice is ranked as a new entry at its own drifted price, so with a competing order the aggressor may trade that order before the next slice. C09's draws hold no other order, so re-ranking cannot change the order of trades here, and C09 does **not** certify it; the reference walk (W) nevertheless ranks by §6.1 at every step, so it states the right rule. Interleaving with a competing level belongs to a scenario that draws one.
- **F12 — v2.14's activation-timing note (not applied here).** v2.14 adds a *Slice activation timing* correction note to §4.2.1 Rule 4: only the "reaches zero" alternative has source support and activation is **eager**, so in the market an iceberg with remaining quantity always shows a slice between matching processes. The same note says the Scope clarification's engineering conclusions are not reopened and that whether a lazily activated slice makes an observable difference is an engineering question for the architect. The r2 correction does not touch (B), so the two-member set of F3 is kept unchanged; the count of each member is reported. **Recorded for the orchestrator:** under the v2.14 note alone, only the first member (slice $m$ shown) is the market outcome.
- **F13 — the canonical texts still describe the observation.** `scenario-ladder.md` §C9 (purpose, *Known result*, the §13.x conformance-gap bullet) and `scenario-ladder-plan.md` §C9 still say the wrong-sign draw is an unasserted observation of a conformance gap. The purpose is copied verbatim above, as the template requires, and the correction paragraph states what changed; the canonical texts are for their owners to amend.
- **F14 — "book unchanged" is read at the rejected submission.** §5.1 puts REJECTED before ACCEPTED, so a rejected order never enters the book; §5.2 gives an order its timestamp on entering the SOB, so a rejected one takes none. The book state before and after submission 0 (both sides, the trade log, the iceberg registry and its count, the timestamp counter) is asserted identical; the final book then differs from the initial empty book only by the aggressor's resting order.
- **F15 — the rejection reason is asserted only as a category.** The record distinguishes a validator rejection from a closed-contract one, and the validator's own reason code is unobserved. Each wrong-sign rejection is asserted to be a **validator** rejection (the contract is open: the aggressor, one timestamp later, is accepted); which validation failed is not observable.

## 2. The rule, quoted

The governing passages, quoted **verbatim** (byte-copied) and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory sources are shown as citations, not machine-checked.

How the quotes ground the expectation of section 5:

- **The price ladder (§4.2).** Each new slice is priced at the previous slice price plus the signed delta: slice $k$ at $\mathit{base}+k\,\mathit{ppd}$; buy deltas negative, sell positive. A slice is activated when the previous one is exhausted; only the peak is shown.
- **Where matching stops (§4.2.1 Rule 2 and its Consequence; §6.2 items 2 and 4).** The aggressor keeps matching regenerated slices until a slice's drifted price no longer crosses its limit; its NON residual then rests.
- **Who sets the price (§6.2 item 7).** The iceberg, on every slice.
- **The scope (§4.2.1 Rule 2 *Re-ranking*, v2.14).** A regenerated slice is ranked by §6.1 as a new entry at its drifted price. With one iceberg and no competing level, the ranking is the ladder itself (finding F11).
- **The sign rule as a rejection condition (§5.4, v2.14; §13.6; §5.1; §5.2).** A BUY ICEwPI with PPD > 0 or a SELL with PPD < 0 is rejected; a rejected order never reaches ACCEPTED, so it never enters the book, trades nothing and takes no timestamp (findings F8, F14).
- **The magnitude (§13.6).** There is no magnitude bound; the envelope is empirical, tied to a dataset; the 2026-01-12 `PPD = -3.00` counter-example (ASCII hyphen).
- **Why the r1 observation was not asserted, and why r2 may assert (OQ-23, CORRECTION + CLOSURE NOTE (a)).** The note bars pinning behaviour on a domain-forbidden input; §5.4 now states the behaviour, so the r2 assertion pins the rule (finding F8).
- **The permitted between-aggressor state (§4.2.1 Scope clarification; OQ-20).** Why the final book of a main draw is one of two listed books (finding F3; see F12).
- **Why the exact sequence (review §16.1, D3-08).** I5 accepts any rung of the declared ladder, so C09 owns the drift order; the counterparty sequence is pinned exactly.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §4.2: the ICEwPI slice price and its sign, the slice mechanism ---
    SpecQuote(
        'market-spec.md',
        '§4.2',
        '**ICEwPI (Iceberg with Price Increment):** Each new slice is priced at the previous slice price ± a fixed delta. Buy icebergs: negative delta (price decreases per slice). Sell icebergs: positive delta (price increases per slice).',
        note='The price ladder: slice k at base + k*ppd, ppd < 0 for a BUY iceberg and > 0 for a SELL iceberg. The drift is certified on sign-valid draws; a sign-invalid increment is rejected (§5.4).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'A regular limit order with hidden total quantity and visible peak quantity.',
        note='The iceberg: a total quantity, of which only a peak is shown.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'Only the peak size is shown in the order book.',
        note='What the final book shows of the iceberg: at most one visible slice; the rest is hidden.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'When a slice is exhausted, a new slice of peak size is activated and receives a new timestamp.',
        note='Slices 0..m-1 are exhausted one after another; slice m is the next one activated.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'The PPD field carries the per-slice price increment (0 for standard icebergs, non-zero for ICEwPI).',
        note='The tape column the increment is read from; every main draw has PPD != 0.',
    ),
    # --- §4.2.1 Rule 2: where matching stops ---
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '**Rule 2 — Multi-slice matching per aggressive order (NON, IOC, FOK; M6.4 update).**\nA single aggressive order (NON, IOC, or FOK) can fill multiple iceberg slices within one match cycle. After the visible peak is consumed, the iceberg regenerates a new slice (at its natural drifted price for ICEwPI)',
        note='One aggressor fills slices 0..m-1 in its own submission, each at its drifted price. (part 1 of 2; the passage is quoted in full, in consecutive parts)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "and matching continues against the same aggressive order until either the aggressive order is fully filled, the iceberg is fully exhausted, or the regenerated slice's price no longer crosses the aggressive order's limit. All slice interactions occur within a single `submit_order` call.",
        note='(continued, part 2 of 2) The stop C09 pins is the third: at slice m, with the aggressor not filled and the iceberg not exhausted.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "*Consequence:* To consume a 10 MW standard iceberg (PPD=0) with peak = 3 MW, a single NON, IOC, or FOK order for 10 MW fills all four slices (3 + 3 + 3 + 1 MW) within one match cycle. For ICEwPI icebergs, matching stops at the slice whose drifted price no longer crosses the aggressive order's limit.",
        note='The stop index m: the first slice whose drifted price does not cross L.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'Subject to §6.1, each regenerated slice is ranked as a new entry at its own (drifted) price. The aggressor always trades the best-ranked counter bid in the book next.',
        note='The scope (finding F11): with one iceberg and no competing level, the best-ranked counter bid is always the next slice of the ladder, so re-ranking is not exercised and not certified here.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "an iceberg whose visible peak was fully consumed by a prior aggressor's matching process and whose next slice has been registered for later regeneration is a permitted between-aggressor state under the matching algorithm (a new slice is activated when the previous one's quantity reaches zero or when subsequent matching draws against the remaining hidden quantity — see",
        note='Why the final book is one of two listed books: slice m visible at its drifted price, or not yet shown (finding F3).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        'Only the first alternative has source support. Activation is **eager**.',
        note="v2.14's activation-timing correction note (finding F12): recorded, not applied, since the r2 correction leaves (B) unchanged; the same note defers the observable question to the architect.",
    ),
    # --- §5.1, §5.2, §5.4: the sign rule as a rejection condition ---
    SpecQuote(
        'market-spec.md',
        '§5.4',
        '**ICEwPI price-increment sign.** An iceberg with a non-zero price increment (PPD) of the wrong sign is rejected: a BUY with PPD > 0, or a SELL with PPD < 0. PPD = 0 is a standard iceberg and is always valid on this check. The magnitude of PPD has no rule-level bound (§13.6 item 2; OQ-13).',
        note='The r2 claim on the wrong-sign draws: the iceberg is REJECTED (finding F8). Every wrong-sign draw has ppd != 0.',
    ),
    SpecQuote(
        'market-spec.md',
        '§5.1',
        'REJECTED (fails OMIE or XBID validation)',
        note='A rejected order never reaches ACCEPTED: it never rests, never trades, and is never cancelled or evicted (lost empty).',
    ),
    SpecQuote(
        'market-spec.md',
        '§5.2',
        'Each order receives a timestamp when entering the SOB.',
        note='A rejected order never enters the SOB, so it takes no timestamp: the timestamp counter is unchanged by its submission (finding F14).',
    ),
    # --- §6.2: crossing, residual, price-setter ---
    SpecQuote(
        'market-spec.md',
        '§6.2 item 2',
        'Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.',
        note='The crossing test, slice by slice: slice k crosses the aggressor limit L iff sell price <= buy price.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 4',
        'If incoming order is partially filled and has NON restriction: residual enters the book with its original entry timestamp (see §5.2).',
        note="The aggressor's residual, q_agg - m*peak, rests at its limit L (on a wrong-sign draw: all of q_agg, the book being empty).",
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 7',
        "Iceberg: when a single incoming order matches multiple slices of the same iceberg, the iceberg is always the price-setter. Source: Continuous Trading Matching Algorithm §5.1.2. This includes the §4.2.1 Rule-3 hop, where a regenerated slice crosses the same aggressor's resting remainder (OQ-31, resolved 2026-10-02).",
        note='Every trade prints at its slice price base + k*ppd, never at the aggressor limit L (they differ for k < m - 1; finding F4).',
    ),
    # --- §13.6: the sign rule (hard) and no magnitude bound ---
    SpecQuote(
        'market-spec.md',
        '§13.6',
        '**Verdict: No numeric magnitude bound exists at the rule level. Only a sign constraint is authoritative.**',
        note="§13.6's verdict.",
    ),
    SpecQuote(
        'market-spec.md',
        '§13.6',
        '**Sign rule (hard).** ICEwPI buy orders must have `PPD < 0`; ICEwPI sell orders must have `PPD > 0`.',
        note='Every main draw is sign-valid and every wrong-sign draw sign-invalid, both checked from the tape (section 4). §5.4 makes the rule a rejection condition.',
    ),
    SpecQuote(
        'market-spec.md',
        '§13.6',
        '`data/rawOMIE/` delivery date 2026-01-12, contract id 81: `PPD = -3.00 EUR/MWh` observed on ICEwPI BUY rows (agent `ISMSL`, unit `ISMSC01`, contracts 81-83). This invalidates the prior approximate-range statement in §13.3.',
        note='The 2026-01-12 counter-example, byte-copied with its ASCII hyphen.',
    ),
    SpecQuote(
        'market-spec.md',
        '§13.6',
        '`orders_20250601.1` sample: `|PPD|` up to approximately 1.55 EUR/MWh.',
        note='The dataset the draw envelope (|ppd| 0.01-0.50 EUR/MWh) is tied to, as an empirical choice (finding F1).',
    ),
    SpecQuote(
        'market-spec.md',
        '§13.6',
        'Empirical `|PPD|` is therefore unbounded from the perspective of the market-spec; test envelopes must be derived empirically from the specific dataset under test, not from a fixed spec constant.',
        note='The draw range is a test envelope, never a bound.',
    ),
    SpecQuote(
        'market-spec.md',
        '§13.6',
        'Validate the sign rule (buy => negative, sell => positive) as a hard constraint.',
        note='Certified in r2 on the wrong-sign draws (both sides): the iceberg is rejected (§5.4).',
    ),
    SpecQuote(
        'market-spec.md',
        '§13.6',
        'Do not encode a fixed numeric magnitude envelope for `PPD` in simulator validation or tests. Any numeric envelope used for robustness/sanity checking must be justified as an empirical choice tied to a specific dataset, not cited as a market rule.',
        note='Why section 3 states the envelope as an empirical choice.',
    ),
    # --- open questions and the domain review ---
    SpecQuote(
        'open-questions.md',
        'OQ-23',
        "That primary probe *does* mechanically construct the gap (the simulator's `validate_order` does not currently inspect PPD sign, so it is accepted at the code boundary), but it is **not valid OMIE order flow** and must **not** be scripted by future readers as an OQ-23 constructor. A locked contract must never pin behavior on domain-forbidden inputs.",
        block='**(a) The PRIMARY ICEwPI-SELL probe',
        note='CORRECTION + CLOSURE NOTE (a): why r1 only observed the wrong-sign draw. In r2 the outcome is a stated rule (§5.4: rejected), so asserting it pins the rule, not behaviour on a forbidden input (finding F8).',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-20',
        "An active iceberg slot whose `order_id` is absent from the LOB at rest between aggressors — i.e. the visible peak was fully consumed by a prior aggressor's matching process and the next slice has not yet been activated — is a **permitted between-aggressor state** under the matching algorithm.",
        block='Ruling.',
        note='The permitted between-aggressor state (finding F3).',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        '§16.1',
        'It is a known residual, not a finding: a slice priced on the wrong rung passes\nI5. C09 owns the drift order.',
        note='Why the exact price sequence is asserted here: I5 accepts any rung of the declared ladder.',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'D3-08',
        "Any scenario that draws two same-price icebergs against one\naggressor (C05's interloper family, C08's multi-slice draw, C09) should pin the exact counterparty\nsequence in its Known result.",
        note='Why every trade row is pinned by buyer and seller order id.',
    ),
)
CITATIONS = (
    Citation(
        'Continuous Trading Matching Algorithm (public description) §3.3 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/Continuous_Trading_Matching_Algorithm_Public_Description.md:274-276',
        'Iceberg orders can be entered with a peak price delta. Each new slice will then be entered with a new limit price which is reduced by the peak price delta for buy orders and increased by the peak price delta for sell orders.',
    ),
    Citation(
        'OMIE Rules, ICEBERG bids with incremental pricing §4.5 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/OMIE_Day-Ahead_and_Intraday_Electricity_Market_Rules_English.md:10276-10281',
        'As an option, “Iceberg” orders can be entered with a price increment. If entered, each new instance will have a new price, which will be calculated as the price of the previous instance plus the price increment. “Iceberg” purchase orders can only be entered with negative price increments, while sell orders can only be entered with positive price increments.',
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"{sum(c.ok for c in QUOTE_CHECKS)} of {len(QUOTE_CHECKS)} quotes match their source")

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`** valid draws of `C09.icewpi_drift` from `SEED0` (the card's default for a random scenario).
- **`WRONG_DRAWS = 20`** valid draws of `C09.icewpi_wrong_sign` from **`WRONG_SIGN_SEED0 = 1_000_000`**, a disjoint seed range, so one seed names one draw. Each is an **asserted rejection** (r2; finding F8). Both iceberg sides must appear among them (the coverage assertion of section 7).
- **Discards.** A seed the generator refuses (`CapacityBreach`), or a tape outside the capacity margin before the run, or a run that the battery's I8 marks `invalid` after it, is **discarded and replaced by the next seed**. Each loop stops at its target of valid draws or after twice as many attempts; the discard rate is reported. An I8 discard of a draw that passed the pre-run margin is also a failed assertion (finding F9). The certificate's requested draws are `DRAWS + WRONG_DRAWS`.
- **`EAGER_DRAWS = 3`.** For each construction, the first `EAGER_DRAWS` **valid** draws also run eager, and each eager record is compared, integer field by integer field, with its jitted record (`compare_records`). On top of these, an eager run is **forced** for the first valid draw of each of the four main classes and of each wrong-sign side. Every class, side and construction **must** have its eager comparison: a missing one is a failed `eager_jit` assertion. Eager runs are chosen among valid draws, so a discarded seed cannot remove them.
- **The four main classes** (each must appear among the valid main draws): iceberg side (BUY, increments negative; SELL, increments positive) × ($m = 1$, or $m \ge 2$, where the price-setter is discriminated: finding F4). The default seed range holds at least 40 draws of every class. **The two wrong-sign sides**: BUY with $\mathit{ppd} > 0$, SELL with $\mathit{ppd} < 0$.
- **The draw envelope is empirical** (§13.6; finding F1): the generator's ranges, $|\mathit{ppd}|$ 1–50 ticks, are a test envelope tied to the §13.6 sample, never a bound. The tick grids come from `OMIEProductConfig()` and serve only to display euros.

In [ ]:
CERT_ID = "C09"                # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500                    # valid draws of C09.icewpi_drift (card: random scenarios default to 500 draws)
SEED0 = 0                      # first seed of C09.icewpi_drift
WRONG_DRAWS = 20               # valid draws of C09.icewpi_wrong_sign, each an asserted rejection (r2)
WRONG_SIGN_SEED0 = 1_000_000   # first seed of C09.icewpi_wrong_sign (a disjoint range)
EAGER_DRAWS = 3                # first EAGER_DRAWS valid draws of each construction also run eager (plus firsts per class)

In [ ]:
import pathlib

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side, Validity

DRIFT, WRONG = "C09.icewpi_drift", "C09.icewpi_wrong_sign"

CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, IOC, FOK, ICE = (int(ExecRestriction.NON), int(ExecRestriction.IOC), int(ExecRestriction.FOK),
                      int(ExecRestriction.ICE))
EXEC_NAME = {NON: "NON", IOC: "IOC", FOK: "FOK", ICE: "ICE"}
GFS = int(Validity.GFS)

SLICE_MARGIN = 64                     # capacity-validity precondition: every iceberg has ceil(total/peak) <= 64
PRICE_INT_MAX = 1_999_800             # §2.3: +9,999 EUR/MWh in the offset encoding (0 is -9,999)
QTY_INT_MIN, QTY_INT_MAX = 1, 9_990   # §2.3: 0.1 MW and 999 MW in quantity ticks

CLASSES = tuple(f"iceberg {s}, {m}" for s in ("BUY", "SELL") for m in ("m = 1", "m >= 2"))
WRONG_CLASSES = ("wrong-sign BUY (ppd > 0)", "wrong-sign SELL (ppd < 0)")

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C09_icewpi_price_drift.ipynb"

print(f"{CERT_ID}: {DRIFT} x {DRAWS} valid draws from seed {SEED0} (at most {2 * DRAWS} attempts)")
print(f"{CERT_ID}: {WRONG} x {WRONG_DRAWS} valid draws from seed {WRONG_SIGN_SEED0} (at most {2 * WRONG_DRAWS} attempts), "
      f"each an asserted rejection")
print(f"eager: first {EAGER_DRAWS} valid draws of each construction + the first of each class: {CLASSES + WRONG_CLASSES}")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW (display only)")
print(f"capacity envelope: {ENVELOPE}; slice margin <= {SLICE_MARGIN} per iceberg")

## 4. Dataset

Each construction emits a two-row GFS tape on one contract: order ids $1, 2$ and tape timestamps $1, 2$ in row order, then pad rows. There is **one iceberg and no competing level**: no other order rests on either side (scope; finding F11). Write $\sigma$ for the iceberg's side and $\bar\sigma$ for the other.

| construction | submission 0 (the iceberg) | submission 1 (the aggressor) |
|---|---|---|
| `C09.icewpi_drift` | $\sigma$ ICE at $\mathit{base}$ (30–90 EUR/MWh), increment $\mathit{ppd}$, $|\mathit{ppd}| \in [1, 50]$ ticks, **sign-valid** ($\sigma$ = SELL: $\mathit{ppd} > 0$; BUY: $\mathit{ppd} < 0$), peak 1.0–8.0 MW, $n = \lceil \mathit{total}/\mathit{peak} \rceil \in [3, 19]$ | $\bar\sigma$ NON, limit $L$ crossing **exactly** the slices $k < m$, $1 \le m \le n - 2$; quantity $q_{agg}$ strictly more than those $m$ slices |
| `C09.icewpi_wrong_sign` | the same, with the increment's sign **flipped**: **sign-invalid** ($\sigma$ = BUY: $\mathit{ppd} > 0$; SELL: $\mathit{ppd} < 0$) — rejected by §5.4 | the same as drawn for the sign-valid increment |

**What section 5 reads.** The expectation reads the **tape rows** through one reference walk, and `draw.params` for the closed form, which is cross-checked against the walk on every draw and against the tape (the precondition below). `draw.expected` is never read.

**The precondition, checked from the tape for every valid draw** (an aggregate assertion in section 7). *Both constructions:* two real rows on one contract, order ids $1, 2$, tape timestamps strictly increasing, both GFS, prices and sizes within §2.3; row 0 is ICE with $0 < \mathit{peak} < \mathit{total}$, $3 \le n \le 64$, $\mathit{ppd} \ne 0$, and every declared slice price $\mathit{base} + k\,\mathit{ppd}$, $k < n$, within §2.3; row 1 is NON on the other side with no peak and no increment; every value equal to its `params` entry. *Main draws:* $\mathit{ppd}$ **sign-valid**; the crossing slices form the prefix $k < m$ with $m$ equal to `params["m"]`, $m \ge 1$; the aggressor's quantity exceeds slices $0..m-1$ (the stop is on price); and the iceberg still holds hidden quantity once slice $m$ is shown. *Wrong-sign draws:* $\mathit{ppd}$ **sign-invalid** (the §5.4 rejection condition holds).

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings), checked from the tape **before** each run, for both constructions: the draw's analytic bounds (`draw.capacity_bounds`, and the generic `tape_capacity_bounds(tape)`) within the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 concurrent icebergs), and the largest $\lceil \mathit{qty}/\mathit{peak} \rceil$ over the tape's iceberg rows ≤ 64. A tape outside it is never run. After the run, the battery's I8 confirms the flags.

Below: the first main draw in readable units beside the integers, its price ladder, the class counts and the envelope actually drawn in the initial seed range; then the first wrong-sign draw and the side counts of the wrong-sign initial range.

In [ ]:
from collections import Counter

from omiecid_simulator.omiecid.encoding import decode_price, decode_qty

C_CONTRACT, C_SIDE, C_PRICE, C_QTY, C_OID = (synth.COL_CONTRACT, synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY,
                                             synth.COL_OID)
C_AGENT, C_TS, C_EXEC, C_VALIDITY, C_PEAK, C_PPD = (synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC,
                                                    synth.COL_VALIDITY, synth.COL_PEAK, synth.COL_PPD)


def real_rows(tape):
    # The real rows (order_id >= 1), in tape (= submission) order.
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def other(side):
    return SELL if int(side) == BUY else BUY


def crosses(rest_side, p_rest, p_in):
    # §6.2 item 2: sell price <= buy price, for a resting order on rest_side and an incoming limit p_in.
    return int(p_rest) <= int(p_in) if int(rest_side) == SELL else int(p_rest) >= int(p_in)


def ceil_div(a, b):
    return -(-int(a) // int(b))


def sign_valid(side, ppd):
    # §4.2 / §13.6: BUY icebergs negative increments, SELL icebergs positive.
    return int(ppd) < 0 if int(side) == BUY else int(ppd) > 0


def max_slices(tape):
    # Largest ceil(qty / peak) over the tape's iceberg rows (0 when there are none).
    ice = [r for r in real_rows(tape) if int(r[C_EXEC]) == ICE]
    return max((ceil_div(r[C_QTY], max(1, int(r[C_PEAK]))) for r in ice), default=0)


def class_of(tape):
    # The class of a main draw, read from its tape (section 3).
    rows = real_rows(tape)
    ice, agg = rows[0], rows[1]
    n = ceil_div(ice[C_QTY], ice[C_PEAK])
    m = 0
    while m < n and crosses(ice[C_SIDE], int(ice[C_PRICE]) + m * int(ice[C_PPD]), agg[C_PRICE]):
        m += 1
    return f"iceberg {side_name(ice[C_SIDE])}, " + ("m = 1" if m == 1 else "m >= 2")


def wrong_class_of(tape):
    # The side of a wrong-sign draw, read from its tape (section 3).
    return WRONG_CLASSES[0] if int(real_rows(tape)[0][C_SIDE]) == BUY else WRONG_CLASSES[1]


DRAWN = {}       # (construction, seed) -> Draw, or None when the generator refused the seed
REFUSED = {}     # (construction, seed) -> the generator's CapacityBreach message


def prepare(construction, seed):
    # Draw one seed (the run loop of section 7 calls this for a re-draw too).
    try:
        d = synth.CONSTRUCTIONS[construction](seed)
    except synth.CapacityBreach as exc:
        d = None
        REFUSED[(construction, seed)] = str(exc)
    DRAWN[(construction, seed)] = d
    return d


for _s in range(SEED0, SEED0 + DRAWS):
    prepare(DRIFT, _s)
for _s in range(WRONG_SIGN_SEED0, WRONG_SIGN_SEED0 + WRONG_DRAWS):
    prepare(WRONG, _s)


def tape_table(draw, title):
    rows = ["| s | oid | side | exec | price (EUR/MWh) | qty (MW) | peak (MW) | ppd (EUR/MWh) | price_int | qty_int | "
            "peak_int | ppd_int | agent | tape ts |", "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for k, r in enumerate(real_rows(draw.tape)):
        rows.append(f"| {k} | {int(r[C_OID])} | {side_name(r[C_SIDE])} | {EXEC_NAME.get(int(r[C_EXEC]), int(r[C_EXEC]))} | "
                    f"{decode_price(int(r[C_PRICE]), CONFIG):.2f} | {decode_qty(int(r[C_QTY]), CONFIG):.1f} | "
                    f"{decode_qty(int(r[C_PEAK]), CONFIG):.1f} | {int(r[C_PPD]) * CONFIG.price_tick:+.2f} | "
                    f"{int(r[C_PRICE])} | {int(r[C_QTY])} | {int(r[C_PEAK])} | {int(r[C_PPD])} | "
                    f"{int(r[C_AGENT])} | {int(r[C_TS])} |")
    return f"**{title}**\n\n" + "\n".join(rows)


FIRST = next((s for s in range(SEED0, SEED0 + DRAWS) if DRAWN[(DRIFT, s)] is not None), None)
if FIRST is None:
    print(f"{DRIFT}: every seed of the initial range was refused by the generator")
else:
    _d = DRAWN[(DRIFT, FIRST)]
    display(Markdown(tape_table(_d, f"{DRIFT}: seed {FIRST} (digest `{_d.digest()}`)")))
    print(f"seed {FIRST} params: {_d.params}")
    print(f"seed {FIRST} capacity bounds: {dict(_d.capacity_bounds)}; largest ceil(qty/peak): {max_slices(_d.tape)}")
print(f"seeds refused by the generator in the initial ranges: "
      f"{ {c: sorted(s for (k, s) in REFUSED if k == c) for c in (DRIFT, WRONG)} }")

In [ ]:
import matplotlib.pyplot as plt

if FIRST is not None:
    _d = DRAWN[(DRIFT, FIRST)]
    _ice, _agg = real_rows(_d.tape)[0], real_rows(_d.tape)[1]
    _n = ceil_div(_ice[C_QTY], _ice[C_PEAK])
    _sgn = 1 if int(_ice[C_SIDE]) == BUY else -1
    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    for k in range(_n):
        p_k = int(_ice[C_PRICE]) + k * int(_ice[C_PPD])
        x_k = min(int(_ice[C_PEAK]), int(_ice[C_QTY]) - k * int(_ice[C_PEAK]))
        hit = crosses(_ice[C_SIDE], p_k, _agg[C_PRICE])
        ax.barh(decode_price(p_k, CONFIG), _sgn * decode_qty(x_k, CONFIG), height=abs(int(_ice[C_PPD])) * 0.006 + 0.004,
                color="tab:green" if _sgn > 0 else "tab:red", alpha=0.9 if hit else 0.25, hatch=None if hit else "//")
        ax.annotate(f"k={k}", (_sgn * decode_qty(x_k, CONFIG), decode_price(p_k, CONFIG)), fontsize=6)
    ax.axhline(decode_price(int(_agg[C_PRICE]), CONFIG), color="black", ls="--", lw=0.9,
               label=f"aggressor {side_name(_agg[C_SIDE])} limit L = {decode_price(int(_agg[C_PRICE]), CONFIG):.2f}")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("slice size (MW): buy right, sell left; solid = crosses L, hatched = does not")
    ax.set_ylabel("slice price base + k*ppd (EUR/MWh)")
    ax.set_title(f"{DRIFT}, seed {FIRST}: the iceberg's price ladder ({side_name(_ice[C_SIDE])}, "
                 f"ppd {int(_ice[C_PPD]) * CONFIG.price_tick:+.2f})", fontsize=9)
    ax.legend(loc="best", fontsize=7)
    plt.tight_layout()
    plt.show()

_main = [d for (c, s), d in DRAWN.items() if c == DRIFT and d is not None]
_classes = Counter(class_of(d.tape) for d in _main)
display(Markdown("**Classes in the initial seed range** (counts)\n\n| class | draws |\n|---|---|\n"
                 + "\n".join(f"| {k} | {_classes.get(k, 0)} |" for k in CLASSES)))
if _main:
    _env = {
        "|ppd| (ticks)": [abs(int(real_rows(d.tape)[0][C_PPD])) for d in _main],
        "base (EUR/MWh)": [decode_price(int(real_rows(d.tape)[0][C_PRICE]), CONFIG) for d in _main],
        "peak (MW)": [decode_qty(int(real_rows(d.tape)[0][C_PEAK]), CONFIG) for d in _main],
        "n = ceil(total/peak)": [max_slices(d.tape) for d in _main],
        "m (crossed slices)": [int(d.params["m"]) for d in _main],
    }
    display(Markdown("**The envelope actually drawn** (empirical test envelope, never a bound: §13.6, finding F1)\n\n"
                     "| quantity | min | max |\n|---|---|---|\n"
                     + "\n".join(f"| {k} | {min(v):g} | {max(v):g} |" for k, v in _env.items())))

FIRST_WRONG = next((s for s in range(WRONG_SIGN_SEED0, WRONG_SIGN_SEED0 + WRONG_DRAWS)
                    if DRAWN[(WRONG, s)] is not None), None)
if FIRST_WRONG is None:
    print(f"{WRONG}: every seed of the initial range was refused by the generator")
else:
    _w = DRAWN[(WRONG, FIRST_WRONG)]
    display(Markdown(tape_table(_w, f"{WRONG} (asserted rejection, §5.4): seed {FIRST_WRONG} "
                                    f"(digest `{_w.digest()}`)")))
    _r0 = real_rows(_w.tape)[0]
    print(f"declared increment {int(_r0[C_PPD])} ticks on a {side_name(_r0[C_SIDE])} iceberg: sign-valid = "
          f"{sign_valid(_r0[C_SIDE], _r0[C_PPD])} (expected False: §5.4 rejects this iceberg)")
_wrong = [d for (c, s), d in DRAWN.items() if c == WRONG and d is not None]
_wsides = Counter(wrong_class_of(d.tape) for d in _wrong)
display(Markdown("**Wrong-sign sides in the initial seed range** (counts)\n\n| side | draws |\n|---|---|\n"
                 + "\n".join(f"| {k} | {_wsides.get(k, 0)} |" for k in WRONG_CLASSES)))

## 5. Known result

**Notation.** Integer ticks throughout. The iceberg (order $o_{ice}$, side $\sigma$) has base price $\mathit{base}$, signed increment $\mathit{ppd}$, peak $\mathit{peak}$ and total $\mathit{total}$; its slices are $k = 0, 1, \ldots, n-1$, $n = \lceil \mathit{total}/\mathit{peak} \rceil$, slice $k$ priced and sized

$$p_k = \mathit{base} + k\,\mathit{ppd}, \qquad x_k = \min\big(\mathit{peak},\ \mathit{total} - k\,\mathit{peak}\big)$$

(§4.2: each new slice at the previous slice price plus the signed delta). The aggressor (order $o_{agg}$, side $\bar\sigma$, NON) has limit $L$ and quantity $q_{agg}$. A trade row is pinned as $[\,\mathit{buyer},\ \mathit{seller},\ \mathit{price},\ \mathit{qty},\ s\,]$: the buyer's and seller's order ids, the price and quantity in ticks, and the submission $s$ that wrote it (finding F7).

**The stop index.** Slice $k$ crosses $L$ iff sell price ≤ buy price (§6.2 item 2). With a sign-valid increment the slices drift **away** from the aggressor, so the crossing slices form a prefix, and

$$m = \min\{\,k : p_k \text{ does not cross } L\,\}, \qquad 1 \le m \le n-2 \ \text{(section 4)}.$$

**(W) The reference walk** (§5.4, §5.1, §4.2, §4.2.1 Rule 2, §6.2 items 2–4 and 7). The real rows are processed in submission order. An incoming ICE row whose $\mathit{ppd}$ has the wrong sign (BUY $> 0$, SELL $< 0$) is **rejected** (§5.4): it is recorded as rejected and leaves the book, the trade list and the entry counter untouched (§5.1, §5.2). Any other incoming order meets the opposite side ranked by price, then entry (§6.1); while it has quantity and the best opposite entry crosses, it trades $f = \min(\text{left}, \text{visible})$ **at the resting entry's price** (for the iceberg, its current slice price $p_k$: §6.2 item 7). A resting slice used up is replaced at once by the iceberg's next slice, at $p_{k+1}$, which the same incoming order keeps matching if it crosses (Rule 2). What is left of a NON order rests at its limit (§6.2 item 4). It is the same walk as C08's, here with $\mathit{ppd} \ne 0$ and the §5.4 sign check. With one iceberg and no competing level, the §6.1 ranking always selects the iceberg's current slice (scope; finding F11).

**The closed form** (from `draw.params`; checked against (W) on every draw). With $F = \sum_{k<m} x_k = m\,\mathit{peak}$ (slices $k < m \le n - 2$ are full size):

$$\text{price sequence} = \big(\mathit{base},\ \mathit{base}+\mathit{ppd},\ \ldots,\ \mathit{base}+(m-1)\,\mathit{ppd}\big), \qquad \text{stop index} = m$$

$$\text{trades} = \big[\,[b,\ s,\ p_k,\ x_k,\ 1]\ \big]_{k=0}^{m-1}$$

with $(b, s)$ the buyer and seller among $o_{ice}, o_{agg}$ (fixed by the sides; D3-08). Filled: $F$ for both. Remaining: $q_{agg} - F > 0$ for the aggressor, $\mathit{total} - F$ for the iceberg (visible + hidden). **(H)** The iceberg's hidden quantity is positive. **(B)** The final book is **one of** (finding F3):

- $\sigma: [[p_m,\ x_m,\ o_{ice}]]$, $\bar\sigma: [[L,\ q_{agg} - F,\ o_{agg}]]$ — slice $m$ shown at its drifted price, exactly $\mathit{base}+m\,\mathit{ppd}$;
- $\sigma: []$, $\bar\sigma: [[L,\ q_{agg} - F,\ o_{agg}]]$ — slice $m$ not yet shown (the permitted between-aggressor state).

No eviction and no cancellation; both orders accepted.

**(S) The sign counterfactual** (finding F5). The walk **without** the §5.4 check (an engine applying the ladder with the drift sign flipped), on the tape with $\mathit{ppd}$ replaced by $-\mathit{ppd}$, must give a **different** trade list on every main draw, or the draw could not catch a drift-sign error. It is asserted, per draw, from the tape alone (no engine).

**What is asserted, per valid main draw** (one combined assertion; expected and observed recorded in full as canonical compact JSON): (A) accepted ids = both, rejected = none; (P) the price sequence and the stop index; (Q) the trade rows in order, field for field; (F) filled and remaining (visible + hidden) per order id, and no order evicted or cancelled (`lost` empty); (H) the iceberg's hidden quantity positive; (B) the final book `[price_int, qty_int, oid]` per side, one of the two listed books. Separately: the dataset precondition, the closed form against (W), the sign counterfactual, the battery, I3 and I5 on every valid draw (`not_applicable` is vacuously satisfied), the capacity checks, coverage and the eager comparisons (section 7).

**(R) The wrong-sign draws: an asserted rejection** (§5.4; §5.1; §5.2; §6.2 items 2 and 4; findings F8, F14, F15). The iceberg $o_{ice}$ (submission 0) is sign-invalid, so it is rejected; the aggressor $o_{agg}$ (submission 1) meets an empty book, crosses nothing, and rests whole at $L$. Per valid wrong-sign draw, from the tape (and the closed form from `draw.params`, checked against (W)):

$$\text{accepted} = [o_{agg}], \qquad \text{rejected} = \big[[0,\ o_{ice},\ \texttt{"validator"}]\big], \qquad \text{trades} = [\,]$$

Filled: $0$ for both. Remaining: $0$ for $o_{ice}$, with no hidden quantity; $q_{agg}$ for $o_{agg}$. `lost` empty (a rejected order is never cancelled or evicted: it was never accepted). **(U)** The book state after submission 0 equals the state before it, field for field: both sides, the trade log, the iceberg registry and its count, and the timestamp counter (a rejected order takes no timestamp). **(B′)** The final book is exactly $\bar\sigma: [[L,\ q_{agg},\ o_{agg}]]$, $\sigma: [\,]$ — a single listed book: with no iceberg, F3's alternative cannot arise.

The code below implements (W) as one function of a tape's rows, and the closed forms (main and wrong-sign) as functions of `draw.params`. None reads `draw.expected`, and none runs the engine.

In [ ]:
import json


def canon(x):
    # Canonical compact JSON: how every per-draw expected / observed value is recorded (in full).
    return json.dumps(x, sort_keys=True, separators=(",", ":"))


def rank_key(side, price, entry):
    # §6.1: SELL lowest price first, BUY highest price first; ties by the earlier entry.
    return ((int(price) if int(side) == SELL else -int(price)), int(entry))


def walk(tape, validate=True):
    # (W): the reference walk over the real rows. Returns the trades, the final book entries and the
    # rejected rows [submission, oid]. validate=False drops the §5.4 sign check (the counterfactual of (S)).
    book, trades, seq, rejected = [], [], [0], []

    def nxt():
        seq[0] += 1
        return seq[0]

    def activate(e):
        # The iceberg's next slice: a new entry at the next drifted price (§4.2).
        ic = e["ice"]
        ic["k"] += 1
        e["vis"] = min(ic["peak"], e["hidden"])
        e["hidden"] -= e["vis"]
        e["price"] = ic["base"] + ic["k"] * ic["ppd"]
        e["seq"] = nxt()

    for s, r in enumerate(real_rows(tape)):
        side, p, q, oid, ex = int(r[C_SIDE]), int(r[C_PRICE]), int(r[C_QTY]), int(r[C_OID]), int(r[C_EXEC])
        ice = ex == ICE
        if validate and ice and int(r[C_PPD]) != 0 and not sign_valid(side, r[C_PPD]):
            rejected.append([s, oid])           # §5.4: never enters the book, takes no entry (§5.1, §5.2)
            continue
        peak = int(r[C_PEAK]) if ice else q
        vis = min(peak, q)
        inc = {"side": side, "price": p, "vis": vis, "hidden": q - vis, "oid": oid, "seq": nxt(), "fresh": False,
               "ice": {"base": p, "ppd": int(r[C_PPD]), "peak": peak, "k": 0} if ice else None}
        while True:
            while inc["vis"] > 0:
                opp = [e for e in book if e["side"] != side and e["vis"] > 0 and crosses(e["side"], e["price"], inc["price"])]
                if not opp:
                    break
                best = min(opp, key=lambda e: rank_key(e["side"], e["price"], e["seq"]))
                f = min(inc["vis"], best["vis"])
                buyer, seller = (oid, best["oid"]) if side == BUY else (best["oid"], oid)
                trades.append([buyer, seller, best["price"], f, s])   # at the resting entry's (slice's) price
                inc["vis"] -= f
                best["vis"] -= f
                best["fresh"] = False
                if best["vis"] == 0:
                    if best["hidden"] > 0:
                        activate(best)          # Rule 2: the next drifted slice, matched in this same submission
                        best["fresh"] = True    # untouched since activation (finding F3)
                    else:
                        book.remove(best)
            if inc["vis"] == 0 and inc["hidden"] > 0:
                activate(inc)
                continue
            break
        if inc["vis"] > 0 and ex in (NON, ICE):
            book.append(inc)                    # §6.2 item 4 / §4.5: what is left rests
    return trades, book, rejected


def book_view(entries, skip=()):
    out = {"BUY": [], "SELL": []}
    for e in entries:
        if e["vis"] > 0 and e["oid"] not in skip:
            out[side_name(e["side"])].append([e["price"], e["vis"], e["oid"]])
    return {k: sorted(v) for k, v in out.items()}


def stop_index_of(tape):
    # m = the first slice whose drifted price does not cross the aggressor's limit L.
    rows = real_rows(tape)
    ice, agg = rows[0], rows[1]
    n = ceil_div(ice[C_QTY], ice[C_PEAK])
    m = 0
    while m < n and crosses(ice[C_SIDE], int(ice[C_PRICE]) + m * int(ice[C_PPD]), agg[C_PRICE]):
        m += 1
    return m


def expectation(draw):
    # Everything section 7 compares for one main draw, from (W) over the tape's rows.
    rows = real_rows(draw.tape)
    oids = [int(r[C_OID]) for r in rows]
    trades, book, _ = walk(draw.tape)
    filled = {o: 0 for o in oids}
    for t in trades:
        filled[t[0]] += t[3]
        filled[t[1]] += t[3]
    remaining = {o: 0 for o in oids}
    hidden = {o: 0 for o in oids}
    for e in book:
        remaining[e["oid"]] += e["vis"] + e["hidden"]
        hidden[e["oid"]] += e["hidden"]
    fresh = [e["oid"] for e in book if e["fresh"]]
    books = [book_view(book)]
    if fresh:                                   # finding F3: a just-activated, untouched slice may not be shown yet
        books.append(book_view(book, skip=set(fresh)))
    ice_oid = int(rows[0][C_OID])
    exact = {
        "accepted": sorted(oids), "rejected": [],
        "trades": trades,
        "price_sequence": [t[2] for t in trades],
        "stop_index": stop_index_of(draw.tape),
        "filled": {str(o): filled[o] for o in oids},
        "remaining": {str(o): remaining[o] for o in oids},
        "hidden_positive": hidden[ice_oid] > 0,
        "lost": {},
    }
    return {"exact": exact, "allowed": {"final_book": books}, "oids": oids, "ice_oid": ice_oid,
            "ice_side": int(rows[0][C_SIDE]), "class": class_of(draw.tape), "n_walk_trades": len(trades)}


def closed_form(p):
    # The closed form of the formulas above, from draw.params only.
    o_i, o_a = int(p["ice_oid"]), int(p["agg_oid"])
    base, ppd, peak, total = int(p["base_int"]), int(p["ppd_int"]), int(p["peak_int"]), int(p["total_int"])
    m, q_agg, L = int(p["m"]), int(p["q_agg_int"]), int(p["p_agg_int"])
    ice_side = int(p["ice_side"])
    b, s = (o_i, o_a) if ice_side == BUY else (o_a, o_i)
    trades = [[b, s, base + k * ppd, min(peak, total - k * peak), 1] for k in range(m)]
    F = sum(t[3] for t in trades)
    ice_s, agg_s = side_name(ice_side), side_name(other(ice_side))
    agg_book = [[L, q_agg - F, o_a]]
    books = [{ice_s: [[base + m * ppd, min(peak, total - F), o_i]], agg_s: agg_book},
             {ice_s: [], agg_s: agg_book}]
    return {"trades": trades, "price_sequence": [base + k * ppd for k in range(m)], "stop_index": m,
            "books": books, "filled": {str(o_i): F, str(o_a): F},
            "remaining": {str(o_i): total - F, str(o_a): q_agg - F}}


def closed_form_agrees(draw):
    try:
        cf, e = closed_form(draw.params), expectation(draw)
    except Exception:  # noqa: BLE001  (a malformed params dict is a disagreement, recorded by the caller)
        return False
    x = e["exact"]
    return bool(cf["trades"] == x["trades"] and cf["price_sequence"] == x["price_sequence"]
                and cf["stop_index"] == x["stop_index"] == e["n_walk_trades"]
                and cf["filled"] == x["filled"] and cf["remaining"] == x["remaining"] and x["hidden_positive"]
                and sorted(map(canon, cf["books"])) == sorted(map(canon, e["allowed"]["final_book"])))


def flipped(tape):
    # The sign counterfactual: the same tape with every iceberg row's increment negated.
    t = np.array(tape, copy=True)
    ice = t[:, C_EXEC] == ICE
    t[ice, C_PPD] = -t[ice, C_PPD]
    return t


def sign_discriminates(draw):
    # (S): a ladder applied with the drift sign flipped (no §5.4 check) gives a different trade list (finding F5).
    return walk(flipped(draw.tape), validate=False)[0] != walk(draw.tape)[0]


def wrong_expectation(draw):
    # (R): everything section 7 compares for one wrong-sign draw, from (W) over the tape's rows.
    rows = real_rows(draw.tape)
    oids = [int(r[C_OID]) for r in rows]
    trades, book, rejected = walk(draw.tape)
    filled = {o: 0 for o in oids}
    for t in trades:
        filled[t[0]] += t[3]
        filled[t[1]] += t[3]
    remaining = {o: 0 for o in oids}
    hidden = {o: 0 for o in oids}
    for e in book:
        remaining[e["oid"]] += e["vis"] + e["hidden"]
        hidden[e["oid"]] += e["hidden"]
    rej_oids = {o for _, o in rejected}
    ice_oid = int(rows[0][C_OID])
    exact = {
        "accepted": sorted(o for o in oids if o not in rej_oids),
        "rejected": [[sub, o, "validator"] for sub, o in rejected],   # finding F15: the category only
        "trades": trades,
        "filled": {str(o): filled[o] for o in oids},
        "remaining": {str(o): remaining[o] for o in oids},
        "ice_hidden": hidden[ice_oid],
        "lost": {},
        "book_unchanged_by_submission_0": [],    # (U): the list of components that differ, expected empty
        "final_book": book_view(book),           # (B'): a single listed book
    }
    return {"exact": exact, "oids": oids, "ice_oid": ice_oid, "ice_side": int(rows[0][C_SIDE]),
            "class": wrong_class_of(draw.tape)}


def wrong_closed_form(p):
    # The closed form of (R), from draw.params only.
    o_i, o_a = int(p["ice_oid"]), int(p["agg_oid"])
    q_agg, L, ice_side = int(p["q_agg_int"]), int(p["p_agg_int"]), int(p["ice_side"])
    book = {side_name(ice_side): [], side_name(other(ice_side)): [[L, q_agg, o_a]]}
    return {"accepted": [o_a], "rejected": [[0, o_i, "validator"]], "trades": [],
            "filled": {str(o_i): 0, str(o_a): 0}, "remaining": {str(o_i): 0, str(o_a): q_agg},
            "ice_hidden": 0, "final_book": book}


def wrong_closed_form_agrees(draw):
    try:
        cf, x = wrong_closed_form(draw.params), wrong_expectation(draw)["exact"]
    except Exception:  # noqa: BLE001  (a malformed params dict is a disagreement, recorded by the caller)
        return False
    return all(canon(cf[k]) == canon(x[k]) for k in cf)


def precondition(draw, wrong=False):
    # Section 4's dataset precondition, from the tape and params. Returns the list of departures.
    # wrong=True: a C09.icewpi_wrong_sign draw (sign-invalid increment; no stop-index conditions).
    bad, t, p = [], np.asarray(draw.tape), draw.params
    rows = real_rows(t)
    if len(rows) != 2:
        return [f"{len(rows)} real rows (expected 2)"]
    ice, agg = rows[0], rows[1]
    if len({int(x[C_CONTRACT]) for x in rows}) != 1:
        bad.append("more than one contract")
    oids, ts = [int(x[C_OID]) for x in rows], [int(x[C_TS]) for x in rows]
    if oids != [1, 2] or not ts[0] < ts[1]:
        bad.append(f"order ids {oids} / tape timestamps {ts} not [1, 2] / strictly increasing")
    if any(not 0 <= int(x[C_PRICE]) <= PRICE_INT_MAX or not QTY_INT_MIN <= int(x[C_QTY]) <= QTY_INT_MAX for x in rows):
        bad.append("price or quantity outside §2.3")
    if any(int(x[C_VALIDITY]) != GFS for x in rows):
        bad.append("a row is not GFS")
    if int(ice[C_EXEC]) != ICE or int(agg[C_EXEC]) != NON or int(agg[C_PEAK]) != 0 or int(agg[C_PPD]) != 0:
        return bad + ["expected row 0 ICE and row 1 NON without a peak or an increment"]
    if int(agg[C_SIDE]) != other(ice[C_SIDE]):
        bad.append("the aggressor is on the iceberg's side")
    total, peak, base, ppd = int(ice[C_QTY]), int(ice[C_PEAK]), int(ice[C_PRICE]), int(ice[C_PPD])
    n = ceil_div(total, peak) if peak > 0 else 0
    if not 0 < peak < total or not 3 <= n <= SLICE_MARGIN:
        bad.append(f"iceberg peak {peak}, total {total}, ceil(total/peak) {n}")
    if ppd == 0:
        bad.append("increment 0: not an ICEwPI")
    elif not wrong and not sign_valid(ice[C_SIDE], ppd):
        bad.append(f"increment {ppd} on a {side_name(ice[C_SIDE])} iceberg is not sign-valid (§13.6)")
    elif wrong and sign_valid(ice[C_SIDE], ppd):
        bad.append(f"increment {ppd} on a {side_name(ice[C_SIDE])} iceberg is sign-valid: not a §5.4 rejection case")
    if any(not 0 <= base + k * ppd <= PRICE_INT_MAX for k in range(max(n, 0))):
        bad.append("a declared slice price base + k*ppd lies outside §2.3")
    tape_vals = {"ice_oid": int(ice[C_OID]), "agg_oid": int(agg[C_OID]), "ice_side": int(ice[C_SIDE]),
                 "agg_side": int(agg[C_SIDE]), "base_int": base, "ppd_int": ppd, "peak_int": peak, "total_int": total,
                 "n_slices": n, "q_agg_int": int(agg[C_QTY]), "p_agg_int": int(agg[C_PRICE])}
    off = sorted(k for k, v in tape_vals.items() if int(p.get(k, -10**9)) != v)
    if off:
        bad.append(f"params != tape on {off}")
    if max_slices(t) > SLICE_MARGIN:
        bad.append(f"an iceberg row with ceil(qty/peak) > {SLICE_MARGIN}")
    if n < 1 or wrong:
        return bad
    m = stop_index_of(t)
    cross = [crosses(ice[C_SIDE], base + k * ppd, agg[C_PRICE]) for k in range(n)]
    if any(cross[m:]):
        bad.append("the crossing slices are not a prefix")
    if m != int(p.get("m", -1)) or not 1 <= m <= n - 1:
        bad.append(f"stop index from the tape {m} != params m {p.get('m')} or outside [1, n - 1]")
    F = sum(min(peak, total - k * peak) for k in range(m))
    if not int(agg[C_QTY]) > F:
        bad.append(f"q_agg {int(agg[C_QTY])} does not exceed the m crossed slices ({F}): the stop is not on price")
    if int(p.get("e_int", -1)) != int(agg[C_QTY]) - F:
        bad.append("params e_int != q_agg - (crossed slices)")
    if not total - F - min(peak, total - F) > 0:
        bad.append("no hidden quantity left once slice m is shown")
    return bad


# ---- the expectation on the first draw --------------------------------------------------------------------------
CF_AGREES = False
if FIRST is not None:
    _d = DRAWN[(DRIFT, FIRST)]
    _e = expectation(_d)
    CF_AGREES = closed_form_agrees(_d)
    print(f"{DRIFT}, seed {FIRST} [{_e['class']}]:")
    print(f"    (P) price sequence (price_int): {_e['exact']['price_sequence']}; stop index m = {_e['exact']['stop_index']}")
    print(f"        in EUR/MWh: {[round(decode_price(x, CONFIG), 2) for x in _e['exact']['price_sequence']]}")
    print(f"    (Q) trades [buyer, seller, price_int, qty_int, submission]: {_e['exact']['trades']}")
    print(f"    (F) filled {_e['exact']['filled']}; remaining {_e['exact']['remaining']}; (H) hidden > 0: "
          f"{_e['exact']['hidden_positive']}")
    print(f"    (B) final book one of: {_e['allowed']['final_book']}")
    print(f"    (S) drift-sign-flipped trades: {walk(flipped(_d.tape), validate=False)[0]} "
          f"(differs: {sign_discriminates(_d)})")
    print(f"    closed form == walk: {CF_AGREES}; precondition departures: {precondition(_d)}")
print(f"closed form agrees with (W) on the first draw: {CF_AGREES}")

# ---- the expectation on the first wrong-sign draw ---------------------------------------------------------------
WRONG_CF_AGREES = False
if FIRST_WRONG is not None:
    _w = DRAWN[(WRONG, FIRST_WRONG)]
    _x = wrong_expectation(_w)
    WRONG_CF_AGREES = wrong_closed_form_agrees(_w)
    print(f"{WRONG}, seed {FIRST_WRONG} [{_x['class']}]:")
    for _k, _v in _x["exact"].items():
        print(f"    (R) {_k}: {_v}")
    print(f"    closed form == walk: {WRONG_CF_AGREES}; precondition departures: {precondition(_w, wrong=True)}")
print(f"closed form agrees with (W) on the first wrong-sign draw: {WRONG_CF_AGREES}")

## 6. Run

### Eager and jitted (wired at K3, r2)

> **Wired at K3 (r2).** `run_draw` below is the battery's record builder on the draw, exactly as the contract states. Sections 1–5 and 7 were not edited (independence rule 2). No generator mode was added: both C09 constructions (`C09.icewpi_drift`, `C09.icewpi_wrong_sign`) are already in `omie_tape_synth.CONSTRUCTIONS` and are run as drawn; the wrong-sign tape is submitted with its sign-invalid increment unchanged, so whether it is rejected is the engine's answer, not the harness's.

Section 7 consumes exactly **one name** from this section. If it were not callable, section 7 would record a failing `section 6 wired` check, run no draw, and the notebook would end `NOT CERTIFIED`.

**`run_draw(draw, *, jit) -> RunRecord`**: the battery's record builder on one draw, i.e. `tests.omiecid.reference.omie_invariants.record_from_draw(draw, jit=jit)`, with no other argument. It serves **both** constructions with the same call. It runs `draw.tape` **as is** (nothing in it may be rewritten; on a wrong-sign draw the sign-invalid increment is submitted unchanged) on one fresh book of an OPEN contract with the default configs (`OMIEProductConfig()`, `OMIEMarketConfig()`), the draw's own analytic capacity bounds (`draw.capacity_bounds`), no cancellations and no gate-closure expiry. `jit=False` runs eager (the builder wraps the run in `jax.disable_jit()`); `jit=True` runs jitted. It may raise; section 7 records a raise as a failed check on the draw.

Section 7 reads these `RunRecord` fields:
- `fills`: each `Fill`'s `submission`, `row_index`, `buyer_oid`, `seller_oid`, `price_int`, `qty_int`, `passive_oid`;
- `accepted` (its keys) and `rejections` (each `Rejection.submission`, `Rejection.oid` and `Rejection.reason`; the reason is the category `"validator"` or `"contract_not_open"` only, finding F15);
- `pre_snapshots[0]` and `post_snapshots[0]`: the book state around submission 0 (`bids`, `asks`, `trades`, `iceberg_meta`, `n_active`, `ts_counter`), for (U) on the wrong-sign draws;
- `filled`, `still_resting`, `hidden_remaining`, `evicted` and `cancelled_or_expired` (each `Cancellation.qty_int`), per order id;
- `final_snapshot.resting(side)`: `(price, qty, oid, agent, sec, nsec)`;
- `observed_capacity`;
- the whole record, through `run_battery` (its per-check `status` and each violation's `field`; I3, I5 and I8 results) and `compare_records`.

Section 7 calls `run_draw(..., jit=True)` on every draw it uses, of both constructions, and `run_draw(..., jit=False)` on the eager selection of section 3 (the first `EAGER_DRAWS` valid draws of each construction and the first of each main class and of each wrong-sign side). A seed beyond the initial range (a re-draw after a discard) is prepared with `prepare(construction, seed)` before the call.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_draw(draw, *, jit) -> RunRecord, per the contract above.
#
# The battery's record builder on one draw, with no other argument:
# record_from_draw runs draw.tape as is, for both constructions (a wrong-sign draw's
# sign-invalid increment is submitted unchanged: the engine decides the rejection) on one fresh book of one OPEN contract, with the default configs
# (OMIEProductConfig(), OMIEMarketConfig()), the draw's own analytic capacity bounds
# (draw.capacity_bounds), no cancellations (no C09 draw schedules one) and no gate-closure
# expiry. jit=False wraps the whole run in jax.disable_jit() (inside build_record);
# jit=True runs it jitted. Every call starts from its own fresh book; the record carries
# pre/post snapshots per submission and rejections (submission, oid, reason category).
# It may raise; section 7 records the raise.
# ---------------------------------------------------------------------------------------
from tests.omiecid.reference.omie_invariants import record_from_draw


def run_draw(draw, *, jit):
    """One run of `draw.tape` (unmodified) on a fresh OPEN book: the battery's record builder."""
    return record_from_draw(draw, jit=jit)


print("run_draw: record_from_draw(draw, jit=jit) on one OPEN contract, default configs, the draw's capacity "
      "bounds, no cancels, no GCT expiry; both constructions; jitted on every draw, eager on the section-3 "
      "selection (both constructions)")

## 7. Certification

The oracle comparison comes first, by order id and in integer ticks, one combined assertion per valid draw of each construction; then the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per valid main draw.** One assertion named by seed and class, whose `expected` and `observed` are the canonical compact JSON (sorted keys) of section 5's values **in full**, never truncated, so an auditor can check every integer from the results JSON alone: accepted and rejected ids; every trade row `[buyer, seller, price_int, qty_int, submission]` in row order; the price sequence and the stop index; filled and remaining per id; the hidden-quantity flag; evicted / cancelled quantities (`lost`, expected empty); the final book. The final book is one of a listed set (finding F3): `expected` holds the whole set under `final_book_one_of`, and the observed book must be one of its members. The assertion's `detail` lists any key that differs, then, **reported and not asserted**, whether slice $m$ was shown and the number of hop-shaped rows. The draw's tape and parameters are kept for a frozen counterexample (`record_case`).

**What is recorded, per valid wrong-sign draw** (r2: an **asserted rejection**, finding F8). One assertion named by seed and side, `expected` and `observed` in full as canonical compact JSON: accepted ids; rejections `[submission, oid, category]`; every trade row (expected none); filled and remaining per id; the iceberg's hidden quantity (expected 0); `lost` (expected empty); **(U)** the list of book components that differ between the states before and after submission 0 (expected empty; on a difference, `detail` carries both states of each differing component in full); the final book. The draw's tape and parameters are kept (`record_case`).

**Aggregate assertions** (every list in full): the section-6 wiring; the closed forms against (W), on the first draw of each construction and on every valid draw; the dataset precondition (section 4) for both constructions, listing every departing seed and its departures; the **sign counterfactual** discriminates on every valid main draw (finding F5); **I3** and **I5** hold on every valid draw of both constructions (failing seeds listed; a `not_applicable` status is vacuously satisfied, only `fail` counts against it); **no capacity flag inside the margin** (finding F9); the **hop bound** ≤ 160 and the largest $\lceil \mathit{qty}/\mathit{peak} \rceil$ ≤ 64 on every valid draw, with the largest values reported; **coverage** — all four main classes and both wrong-sign sides among the valid draws; each construction reaches its target of valid draws; the eager comparisons, at least one per class, per side and per construction, a missing one recorded as a failure.

The battery runs on every jitted record and is folded into the battery table for both constructions: `expects_trade = True` on main draws (every main draw trades), `expects_trade = False` on wrong-sign draws (none does). A draw that I8 marks `invalid` is discarded and replaced by the next seed. **Reported, not asserted:** the valid draws per class and side, the discards per reason, the eager seeds, how often slice $m$ was shown, and the hop-shaped rows observed.

In [ ]:
import traceback
from collections import defaultdict

from tests.omiecid.reference.omie_invariants import compare_records, run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Price-incremented icebergs",
    known_result="a resting ICEwPI iceberg (sign-valid ppd != 0: BUY < 0, SELL > 0) and one NON aggressor whose limit L "
                 "crosses exactly the first m slices and whose quantity exceeds them: trades [buyer, seller, base + k*ppd, "
                 "peak, 1] for k = 0..m-1, so the exact price sequence base, base+ppd, ..., base+(m-1)ppd and the stop "
                 "index m; the aggressor's residual rests at L; the iceberg keeps total - m*peak with positive hidden "
                 "quantity, its slice m shown at exactly base + m*ppd or not yet shown (OQ-20); the iceberg sets every "
                 "price (§6.2 item 7). One iceberg, no competing level. Every wrong-sign draw (BUY ppd > 0, SELL ppd < 0) "
                 "is REJECTED (§5.4 v2.14; §13.6 sign rule): no trade, the book unchanged by its submission, and the "
                 "aggressor resting whole at L",
    oracle_kind="exact-parametric", quotes=QUOTES, citations=CITATIONS, constructions=(DRIFT, WRONG),
    requested_draws=DRAWS + WRONG_DRAWS, runtime_forecast_s=300.0, started_at=T0,
    downstream="C10 (C10c's oracle is C09's price sequence summed: the FOK pre-check's ladder projection), C11 (GCT "
               "expiry of resting ICEwPI icebergs), C15 (fuzz sessions with ICEwPI regimes); invariant I5, whose "
               "ladder exemption accepts any rung (C09 owns the drift order); Track B, whose replayed books contain "
               "ICEwPI icebergs",
)

R_SEQ = "§4.2 ICEwPI (slice k at base + k*ppd); §4.2.1 Rule 2 (stop at the first non-crossing slice); §6.2 items 2, 4, 7"
R_I3 = "§4.2.1 Rule 4 (a crossed order book cannot persist); I3"
R_I5 = "§6.2 item 7 (the iceberg is the price-setter); I5"
R_CAP = "capacity-validity precondition (five ceilings)"
R_DATA = "dataset precondition (section 4); §2.3; §13.6 sign rule (sign-valid main draws, sign-invalid wrong-sign draws)"
R_EAGER = "eager == jitted (scenario-ladder: what eager and jitted mean here)"
R_REJ = ("§5.4 ICEwPI price-increment sign (v2.14): a BUY with PPD > 0 or a SELL with PPD < 0 is rejected; §13.6 sign "
         "rule; §5.1 REJECTED; §5.2 (no timestamp outside the SOB); §6.2 items 2 and 4 (the aggressor rests whole)")


def guarded(label, seed, fn, *args, **kw):
    # Call a section-6 function on a main draw; a raise is recorded as a failed check, never propagated.
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a RunRecord", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_SEQ)
        return None


def book3(snap):
    return {side_name(sd): sorted([int(x[0]), int(x[1]), int(x[2])] for x in snap.resting(sd)) for sd in (BUY, SELL)}


def trade_rows(rec):
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    return fills, [[int(f.buyer_oid), int(f.seller_oid), int(f.price_int), int(f.qty_int), int(f.submission)]
                   for f in fills]


def lost_of(rec):
    return {str(x): [int(rec.evicted.get(x, 0)), sum(int(c.qty_int) for c in rec.cancelled_or_expired.get(x, ()))]
            for x in sorted(set(rec.evicted) | set(rec.cancelled_or_expired))
            if int(rec.evicted.get(x, 0)) or sum(int(c.qty_int) for c in rec.cancelled_or_expired.get(x, ()))}


def observe(rec, e):
    # The observed counterpart of e["exact"] and e["allowed"], plus the reported-only info.
    fills, trades = trade_rows(rec)
    oids, ice = e["oids"], e["ice_oid"]
    o = {
        "accepted": sorted(int(x) for x in rec.accepted),
        "rejected": sorted(int(r.oid) for r in rec.rejections),
        "trades": trades,
        "price_sequence": [t[2] for t in trades],
        "stop_index": len(trades),             # finding F6: one row per crossed slice on this geometry
        "filled": {str(x): int(rec.filled.get(x, 0)) for x in oids},
        "remaining": {str(x): int(rec.still_resting.get(x, 0)) + int(rec.hidden_remaining.get(x, 0)) for x in oids},
        "hidden_positive": int(rec.hidden_remaining.get(ice, 0)) > 0,
        "lost": lost_of(rec),
        "final_book": book3(rec.final_snapshot),
    }
    shown = any(r[2] == ice for r in o["final_book"][side_name(e["ice_side"])])
    sub_oid = {s: oids[s] for s in range(len(oids))}
    info = {"slice_m": "shown" if shown else "not shown",
            "hop_shaped_rows": sum(1 for f in fills if int(f.passive_oid) == sub_oid.get(int(f.submission)))}
    return o, info


def check_draw(seed, rec, e):
    # (A), (P), (Q), (F), (H), (B) of section 5, on the jitted record. Returns (ok, info).
    o, info = observe(rec, e)
    diff = [k for k, v in e["exact"].items() if o.get(k) != v]
    diff += [k for k, allowed in e["allowed"].items() if o.get(k) not in allowed]
    expected = {**e["exact"], **{f"{k}_one_of": v for k, v in e["allowed"].items()}}
    ok = cert.check(
        f"seed {seed} [{e['class']}]: exact price sequence, stop index, trade rows, fills, hidden, final book",
        not diff, seed=seed, expected=canon(expected), observed=canon(o), unit="other",
        detail=("differs on: " + ", ".join(diff) + " | " if diff else "") + "reported only: " + canon(info),
        rule=R_SEQ)
    return ok, info


EAGER_DONE = defaultdict(list)   # class or construction -> seeds with an eager comparison


def eager_compare(seed, draw, rec, tags):
    # Run the draw eager and compare it with its jitted record (one eager_jit assertion).
    rec_e = guarded(f"seed {seed} (eager)", seed, run_draw, draw, jit=False)
    diff = (["jitted run missing" if rec is None else "eager run missing"] if rec is None or rec_e is None
            else list(compare_records(rec_e, rec)))
    for t in tags:
        EAGER_DONE[t].append(seed)
    cert.check(f"eager == jitted (seed {seed})", diff == [], kind="eager_jit", seed=seed,
               expected=True, observed=diff == [], unit="bool", detail=", ".join(diff) if diff else "identical",
               rule=R_EAGER)


def margin_breaches(tape, bounds):
    # The capacity-validity precondition (five ceilings), checked from the tape before it is run.
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    bad = []
    for name, bd in (("draw", bounds), ("generic", synth.tape_capacity_bounds(tape))):
        bad += [f"{name} {k} bound {bd.get(k)} > {m}" for k, m in lim.items() if int(bd.get(k, 0)) > m]
    if max_slices(tape) > SLICE_MARGIN:
        bad.append(f"ceil(qty/peak) {max_slices(tape)} > {SLICE_MARGIN}")
    return bad


cert.check("section 5: closed form agrees with the walk (W) on the first draw", CF_AGREES,
           expected=True, observed=CF_AGREES, unit="bool", rule=R_SEQ)
cert.check("section 5 (R): closed form agrees with the walk (W) on the first wrong-sign draw", WRONG_CF_AGREES,
           expected=True, observed=WRONG_CF_AGREES, unit="bool", rule=R_REJ)
wired = callable(globals().get("run_draw"))
cert.check("section 6 wired (run_draw defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_SEQ)

PER_RUN, DATA_BAD, CLOSED_BAD, NO_DISCRIM = [], {}, [], []
VALID = []
COVER = Counter()
FLAG_INSIDE_MARGIN = {}                    # seed -> I8 evidence, for draws that passed the pre-run margin
BAT_FAIL = defaultdict(list)               # check id -> failing seeds (I3, I5)
DISCARDS = Counter()
PEAK = {"hop_bound": 0, "generic_hop_bound": 0, "max_slices": 0}
SHOWN = Counter()
HOPS_SEEN = 0


def run_main():
    global HOPS_SEEN
    valid, seed = 0, SEED0
    while valid < DRAWS and seed < SEED0 + 2 * DRAWS:
        draw = DRAWN.get((DRIFT, seed), "missing")
        if draw == "missing":
            draw = prepare(DRIFT, seed)
        if draw is None:
            cert.discard(seed, f"{DRIFT}: generator refused the seed: {REFUSED.get((DRIFT, seed))}")
            DISCARDS["generator refused"] += 1
            seed += 1
            continue
        margin = margin_breaches(draw.tape, dict(draw.capacity_bounds))
        if margin:
            cert.discard(seed, f"{DRIFT}: pre-run capacity margin: " + "; ".join(margin))
            DISCARDS["pre-run margin"] += 1
            seed += 1
            continue
        bad = precondition(draw)
        if bad:
            DATA_BAD[f"seed {seed}"] = bad
            cert.check(f"seed {seed}: draw is as section 4 describes", False, seed=seed,
                       expected=[], observed=bad, unit="other", rule=R_DATA)
            seed += 1
            continue
        e = expectation(draw)
        rec = guarded(f"seed {seed} (jit)", seed, run_draw, draw, jit=True)
        if rec is None:
            seed += 1
            continue
        rep = run_battery(rec, expects_trade=True)
        if not cert.add_battery(rep, seed=seed):
            i8 = next(r for r in rep.results if r.check_id == "I8")
            FLAG_INSIDE_MARGIN[f"seed {seed}"] = dict(i8.evidence)
            DISCARDS["I8 invalid"] += 1
            seed += 1   # discarded and re-drawn with the next seed (and a failed assertion: finding F9)
            continue
        valid += 1
        VALID.append(seed)
        cert.record_case(seed, draw.tape, {**draw.params, "construction": DRIFT})
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["generic_hop_bound"] = max(PEAK["generic_hop_bound"], int(synth.tape_capacity_bounds(draw.tape)["hops"]))
        PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(draw.tape))
        COVER[e["class"]] += 1
        if not closed_form_agrees(draw):
            CLOSED_BAD.append(seed)
        if not sign_discriminates(draw):
            NO_DISCRIM.append(seed)
        for r in rep.results:
            if r.check_id in ("I3", "I5") and r.status == "fail":   # not_applicable is vacuously satisfied
                BAT_FAIL[r.check_id].append(seed)
        cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
        ok, info = check_draw(seed, rec, e)
        PER_RUN.append({"seed": seed, "class": e["class"], "ok": ok})
        SHOWN[info["slice_m"]] += 1
        HOPS_SEEN += info["hop_shaped_rows"]
        if valid <= EAGER_DRAWS or e["class"] not in EAGER_DONE:
            eager_compare(seed, draw, rec, [e["class"], DRIFT])
        seed += 1
    return valid


N_VALID = run_main() if wired else 0

# ---- aggregate assertions (every list in full) ----------------------------------------------------------------
cert.check("section 5: closed form agrees with the walk (W) on every valid main draw", wired and not CLOSED_BAD,
           expected=[], observed=CLOSED_BAD, unit="other", rule=R_SEQ)
cert.check("dataset: every main draw is as section 4 describes (precondition, sign-valid)", wired and not DATA_BAD,
           expected={}, observed=DATA_BAD, unit="other",
           detail=f"{len(DATA_BAD)} draws depart" if DATA_BAD else f"{N_VALID} valid draws", rule=R_DATA)
cert.check("section 5 (S): the drift-sign-flipped ladder gives a different trade list on every valid main draw (F5)",
           wired and N_VALID > 0 and not NO_DISCRIM, expected=[], observed=NO_DISCRIM, unit="other",
           detail=f"valid draws: {N_VALID}", rule=R_SEQ + "; §13.6 sign rule")
_got = {c: COVER[c] for c in CLASSES}
_missing = [c for c, v in _got.items() if v == 0]
cert.check("dataset: the valid main draws cover every class (section 3)", wired and not _missing,
           expected={c: ">= 1" for c in CLASSES}, observed=_got, unit="count",
           detail=f"missing: {_missing}" if _missing else "", rule=R_DATA)

# ---- the wrong-sign draws (r2, finding F8): each an ASSERTED rejection ----------------------------------------
SNAP_FIELDS = ("bids", "asks", "trades", "iceberg_meta", "n_active", "ts_counter")


def snap_state(snap):
    # One book state, field for field, as plain lists / ints (for (U)).
    return {f: (np.asarray(getattr(snap, f)).tolist() if f in ("bids", "asks", "trades", "iceberg_meta")
                else int(getattr(snap, f))) for f in SNAP_FIELDS}


def observe_wrong(rec, x):
    # The observed counterpart of x["exact"] for a wrong-sign draw; (U) compares the states around submission 0.
    _, trades = trade_rows(rec)
    oids, ice = x["oids"], x["ice_oid"]
    pre, post = snap_state(rec.pre_snapshots[0]), snap_state(rec.post_snapshots[0])
    changed = [f for f in SNAP_FIELDS if canon(pre[f]) != canon(post[f])]
    o = {
        "accepted": sorted(int(k) for k in rec.accepted),
        "rejected": sorted([int(r.submission), int(r.oid), str(r.reason)] for r in rec.rejections),
        "trades": trades,
        "filled": {str(k): int(rec.filled.get(k, 0)) for k in oids},
        "remaining": {str(k): int(rec.still_resting.get(k, 0)) + int(rec.hidden_remaining.get(k, 0)) for k in oids},
        "ice_hidden": int(rec.hidden_remaining.get(ice, 0)),
        "lost": lost_of(rec),
        "book_unchanged_by_submission_0": changed,
        "final_book": book3(rec.final_snapshot),
    }
    evidence = {f: {"before": pre[f], "after": post[f]} for f in changed}
    return o, evidence


def check_wrong(seed, rec, x):
    # (R), (Z), (U), (F'), (B') of section 5, on the jitted record. Returns ok.
    o, evidence = observe_wrong(rec, x)
    diff = [k for k, v in x["exact"].items() if canon(o.get(k)) != canon(v)]
    return cert.check(
        f"seed {seed} [{x['class']}]: iceberg REJECTED, zero trades, book unchanged by submission 0, aggressor rests whole",
        not diff, seed=seed, expected=canon(x["exact"]), observed=canon(o), unit="other",
        detail=("differs on: " + ", ".join(diff) if diff else "as section 5 (R) states")
               + (" | book components changed by submission 0 (before / after, in full): " + canon(evidence)
                  if evidence else ""),
        rule=R_REJ)


W_VALID, W_DATA_BAD, W_CLOSED_BAD, W_PER_RUN = [], {}, [], []
W_COVER = Counter()


def run_wrong():
    valid, seed = 0, WRONG_SIGN_SEED0
    while valid < WRONG_DRAWS and seed < WRONG_SIGN_SEED0 + 2 * WRONG_DRAWS:
        draw = DRAWN.get((WRONG, seed), "missing")
        if draw == "missing":
            draw = prepare(WRONG, seed)
        if draw is None:
            cert.discard(seed, f"{WRONG}: generator refused the seed: {REFUSED.get((WRONG, seed))}")
            DISCARDS["wrong-sign: generator refused"] += 1
            seed += 1
            continue
        margin = margin_breaches(draw.tape, dict(draw.capacity_bounds))
        if margin:
            cert.discard(seed, f"{WRONG}: pre-run capacity margin: " + "; ".join(margin))
            DISCARDS["wrong-sign: pre-run margin"] += 1
            seed += 1
            continue
        bad = precondition(draw, wrong=True)
        if bad:
            W_DATA_BAD[f"seed {seed}"] = bad
            cert.check(f"seed {seed}: wrong-sign draw is as section 4 describes", False, seed=seed,
                       expected=[], observed=bad, unit="other", rule=R_DATA)
            seed += 1
            continue
        x = wrong_expectation(draw)
        rec = guarded(f"seed {seed} (jit)", seed, run_draw, draw, jit=True)
        if rec is None:
            seed += 1
            continue
        rep = run_battery(rec, expects_trade=False)
        if not cert.add_battery(rep, seed=seed):
            i8 = next(r for r in rep.results if r.check_id == "I8")
            FLAG_INSIDE_MARGIN[f"seed {seed}"] = dict(i8.evidence)
            DISCARDS["wrong-sign: I8 invalid"] += 1
            seed += 1   # discarded and re-drawn with the next seed (and a failed assertion: finding F9)
            continue
        valid += 1
        W_VALID.append(seed)
        cert.record_case(seed, draw.tape, {**draw.params, "construction": WRONG})
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["generic_hop_bound"] = max(PEAK["generic_hop_bound"], int(synth.tape_capacity_bounds(draw.tape)["hops"]))
        PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(draw.tape))
        W_COVER[x["class"]] += 1
        if not wrong_closed_form_agrees(draw):
            W_CLOSED_BAD.append(seed)
        for r in rep.results:
            if r.check_id in ("I3", "I5") and r.status == "fail":   # not_applicable is vacuously satisfied
                BAT_FAIL[r.check_id].append(seed)
        cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
        ok = check_wrong(seed, rec, x)
        W_PER_RUN.append({"seed": seed, "class": x["class"], "ok": ok})
        if valid <= EAGER_DRAWS or x["class"] not in EAGER_DONE:
            eager_compare(seed, draw, rec, [x["class"], WRONG])
        seed += 1
    return valid


N_WRONG = run_wrong() if wired else 0

cert.check("section 5 (R): closed form agrees with the walk (W) on every valid wrong-sign draw",
           wired and not W_CLOSED_BAD, expected=[], observed=W_CLOSED_BAD, unit="other", rule=R_REJ)
cert.check("dataset: every wrong-sign draw is as section 4 describes (precondition, sign-invalid)",
           wired and not W_DATA_BAD, expected={}, observed=W_DATA_BAD, unit="other",
           detail=f"{len(W_DATA_BAD)} draws depart" if W_DATA_BAD else f"{N_WRONG} valid draws", rule=R_DATA)
_wgot = {c: W_COVER[c] for c in WRONG_CLASSES}
_wmissing = [c for c, v in _wgot.items() if v == 0]
cert.check("dataset: the valid wrong-sign draws cover both iceberg sides (section 3)", wired and not _wmissing,
           expected={c: ">= 1" for c in WRONG_CLASSES}, observed=_wgot, unit="count",
           detail=f"missing: {_wmissing}" if _wmissing else "", rule=R_DATA)
for _t, _n, _want in ((DRIFT, N_VALID, DRAWS), (WRONG, N_WRONG, WRONG_DRAWS)):
    cert.check(f"{_t}: {_want} valid draws reached", wired and _n == _want, expected=_want, observed=_n,
               unit="count", rule=R_DATA)

# ---- I3 / I5 over both constructions (only "fail" counts; not_applicable is vacuously satisfied) ---------------
for _cid, _rule in (("I3", R_I3), ("I5", R_I5)):
    cert.check(f"{_cid} holds on every valid draw of both constructions (not_applicable = vacuously satisfied)",
               wired and N_VALID > 0 and N_WRONG > 0 and not BAT_FAIL[_cid],
               expected=[], observed=sorted(BAT_FAIL[_cid]), unit="other",
               detail=f"valid draws: {N_VALID} main, {N_WRONG} wrong-sign", rule=_rule)
cert.check("no capacity flag (truncated, book_full, saturated ring) on a draw inside the analytic margin (F9)",
           wired and not FLAG_INSIDE_MARGIN, expected={}, observed=FLAG_INSIDE_MARGIN, unit="other", rule=R_CAP)
cert.check("hop bound <= 160 on every valid draw (largest reported: draw bound, generic bound)",
           wired and PEAK["hop_bound"] <= ENVELOPE.max_hops and PEAK["generic_hop_bound"] <= ENVELOPE.max_hops,
           expected=f"<= {ENVELOPE.max_hops}", observed=[PEAK["hop_bound"], PEAK["generic_hop_bound"]],
           unit="count", rule=R_CAP)
cert.check("largest ceil(qty/peak) over every valid draw's icebergs <= 64 (largest reported)",
           wired and 0 < PEAK["max_slices"] <= SLICE_MARGIN, expected=f"<= {SLICE_MARGIN}",
           observed=PEAK["max_slices"], unit="count", rule=R_CAP)

# ---- eager: every class, side and construction must have had one comparison (absence = failure) ---------------
for _t in list(CLASSES) + list(WRONG_CLASSES) + [DRIFT, WRONG]:
    if not EAGER_DONE.get(_t):
        cert.check(f"eager == jitted for '{_t}'", False, kind="eager_jit", expected=True, observed=None, unit="bool",
                   detail=f"no eager comparison was run for '{_t}'", rule=R_EAGER)

# ---- reports (not asserted) ------------------------------------------------------------------------------------
_fail_runs = [r for r in PER_RUN if not r["ok"]]
display(Markdown(f"**Per-draw summary.** {len(PER_RUN)} main draws checked, {len(_fail_runs)} with a failed assertion"
                 + (": " + ", ".join(f"seed {r['seed']}" for r in _fail_runs) if _fail_runs else ".")))
_wfail = [r for r in W_PER_RUN if not r["ok"]]
display(Markdown(f"**Wrong-sign summary.** {len(W_PER_RUN)} wrong-sign draws checked, {len(_wfail)} with a failed "
                 "assertion" + (": " + ", ".join(f"seed {r['seed']}" for r in _wfail) if _wfail else ".")))
print(f"valid main draws: {N_VALID}; per class: {dict(COVER)}")
print(f"valid wrong-sign draws: {N_WRONG}; per side: {dict(W_COVER)}")
print(f"discards per reason: {dict(DISCARDS)}")
print(f"eager comparisons run: { {k: v for k, v in EAGER_DONE.items()} }")
print(f"slice m at the end (finding F3; reported only): {dict(SHOWN)}")
print(f"hop-shaped rows observed (passive = the submitting order's own id; reported only): {HOPS_SEEN}")
print(f"analytic peaks over valid draws: hop bound {PEAK['hop_bound']} (generic {PEAK['generic_hop_bound']}), "
      f"largest ceil(qty/peak) {PEAK['max_slices']}")
display(Markdown(cert.table()))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))